# Capstone pipeline

Full Capstone project requires:
1. Download DeepFashion images to train recognition of cloth type and condition.
2. Download Poshmark and Depop sample data for a few cloth types: price, brand, type, condition
3. Clean (remove missing) and unify (rename attributes to be the same across stores) data
4. Prepare data (extract features, prepare training, validation and testing datasets, as well as create synthetic data to similate conditions)
5. Train (run visual training, run categorical training)

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


# Finding where the notebook is located. The path is used for some more complicated script calls instead of making this notebook overgrown.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'README.md').exists() and (PROJECT_ROOT.parent / 'README.md').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

sys.path.append('scripts')
sys.path.append('src')
PYTHON = sys.executable

def run(args, check=True, env=None):
    if isinstance(args, str):
        print(f"$ {args}")
        return subprocess.run(args, shell=True, check=check, env=env)
    cmd = ' '.join(shlex.quote(str(part)) for part in args)
    print(f"$ {cmd}")
    return subprocess.run([str(part) for part in args], check=check, env=env)

print(PROJECT_ROOT)
print(PYTHON)


In [2]:
KAGGLE_DATASETS = [
    ('thusharanair/deepfashion2-original-with-dataframes', 'data/raw/deepfashion/kaggle_thushan'),
    ('vishalbsadanand/deepfashion-1', 'data/raw/deepfashion/kaggle_vishal'),
]

SCRAPE_JOBS = [
    {'query': 'dress', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'pants', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'shorts', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'jeans', 'max_items': 500, 'rate_limit': 2.0},
    {'query': 'skirt', 'max_items': 500, 'rate_limit': 2.0},
]

SYNTHETIC_INPUT_DIR = 'data/deepfashion/original'
SYNTHETIC_OUTPUT_DIR = 'data/deepfashion/synthetic_degraded'
SYNTHETIC_VARIATIONS = 5
SYNTHETIC_MAX_IMAGES = 5000

CLOTHING_TYPE_DIR = 'data/processed/clothing_type'
CONDITION_DIR = 'data/processed/condition_assessment'
MULTITASK_DIR = 'data/processed/multitask'
FEATURES_DIR = 'data/features'
EMBEDDINGS_DIR = 'data/embeddings'
PRICE_DATASET_DIR = 'data/price_classification'
VECTOR_INDEX_DIR = 'data/vector_index'

UNIFIED_DATA_CSV = 'data/scraped/unified/unified_data.csv'

CLOTHING_TYPE_MODEL_DIR = 'models/clothing_type'
CONDITION_MODEL_DIR = 'models/condition_assessment'
MULTITASK_MODEL_DIR = 'models/multitask'
PRICE_MODEL_DIR = 'models/price_model'

VISION_MODEL_FOR_FEATURES = f'{MULTITASK_MODEL_DIR}/checkpoints/best_loss.pth'

TRAIN_DEVICE = 'cuda'

In [6]:
#run(["pip", "install", "gdown"])
#run(["pip", "install", "playwright"])
#run([PYTHON, '-m', 'playwright', 'install', 'chromium'])
#run(["pip", "install", "torch"])

$ pip install torch


CompletedProcess(args=['pip', 'install', 'torch'], returncode=0)

## Warning! Next operation will download ~56GBs of images

## Download

In [3]:
import download_deepfashion
import download_kaggle_datasets
import process_deepfashion_to_csv
import generate_statistics

# This takes several hours to receive all data

#download_deepfashion.main(['--subset', 'all', '--download', '--extract', '--verify']) # this can only be used with signed in google account. commenting it out for public repo.
# Don't forget to set KAGGLE_USERNAME and KAGGLE_KEY in system environment variables
for dataset, output_dir in KAGGLE_DATASETS:
    download_kaggle_datasets.main(['--dataset', dataset, '--output', output_dir])

process_deepfashion_to_csv.main(['--input', 'data/raw/deepfashion', '--output', 'data/processed'])
generate_statistics.main(['--processed-dir', 'data/processed', '--output', 'data/processed/deepfashion_statistics.csv'])

2026-03-26 00:03:15,627 - process_deepfashion_to_csv - INFO - Starting DeepFashion dataset processing...
2026-03-26 00:03:15,627 - process_deepfashion_to_csv - INFO - Input directory: data\raw\deepfashion
2026-03-26 00:03:15,627 - process_deepfashion_to_csv - INFO - Output directory: data\processed
2026-03-26 00:03:15,628 - process_deepfashion_to_csv - INFO - Processing Kaggle Thushan (DeepFashion2) dataset...
2026-03-26 00:03:15,628 - process_deepfashion_to_csv - INFO - Looking for CSV dataframes...
2026-03-26 00:03:19,397 - process_deepfashion_to_csv - INFO - Processing training data from: train.csv
2026-03-26 00:03:21,733 - process_deepfashion_to_csv - INFO - Loaded 312186 training samples
2026-03-26 00:03:21,734 - process_deepfashion_to_csv - INFO - Columns: ['path', 'segmentation', 'landmarks', 'b_box', 'category_id', 'category_name', 'scale', 'viewpoint', 'occlusion', 'zoom_in', 'img_height', 'img_width', 'split', 'source_dataset', 'subset']
2026-03-26 00:03:21,734 - process_deep

0

## Scrape

In [ ]:
import scrape_marketplaces

# This takes several hours to receive all data

for job in SCRAPE_JOBS:
    scrape_marketplaces.main(['--query', job['query'], '--all-platforms', '--max-items', str(job['max_items']), '--rate-limit', str(job['rate_limit']), '--output-dir', 'data/raw/scraped'])

2026-03-25 22:27:27,285 - PoshmarkScraper - INFO - Starting Poshmark search scrape: 'dress' (max: 500)
2026-03-25 22:27:27,286 - PoshmarkScraper - INFO - Scraping page 1...


Starting scrape: POSHMARK
Query: dress
Max items: 500
Rate limit: 2.0s between requests


2026-03-25 22:27:28,500 - PoshmarkScraper - INFO - Playwright browser initialized
2026-03-25 22:27:35,552 - PoshmarkScraper - WARNING - Timeout waiting for selector: [class*="tile"], [class*="card"], [class*="item"]
2026-03-25 22:27:35,791 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Mini-Dress-69c474c1013d2ab2342425fd
2026-03-25 22:27:41,489 - PoshmarkScraper - INFO - Successfully parsed item 69c474c1013d2ab2342425fd: Mini Dress
2026-03-25 22:27:41,491 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Mini-Dress-69c474c1013d2ab2342425fd
2026-03-25 22:27:47,028 - PoshmarkScraper - INFO - Successfully parsed item 69c474c1013d2ab2342425fd: Mini Dress
2026-03-25 22:27:47,029 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Mini-Dress-69c474c1013d2ab2342425fd
2026-03-25 22:27:52,495 - PoshmarkScraper - INFO - Successfully parsed item 69c474c1013d2ab2342425fd: Mini Dress
2026-03-25 22:27:52,496 - PoshmarkScraper - INFO - S


 Failed to scrape poshmark: 'PoshmarkScraper' object has no attribute '_browser_thread'

Starting scrape: THREDUP
Query: dress
Max items: 500
Rate limit: 2.0s between requests


2026-03-25 22:53:52,756 - ThredUpScraper - INFO - Playwright browser initialized
2026-03-25 22:53:52,809 - ThredUpScraper - INFO - GET https://www.thredup.com/search/dress?text=dress&page=1
2026-03-25 22:53:53,599 - ThredUpScraper - INFO - Status=200 Title=Used Dress On Sale Up To 90% Off | ThredUp
2026-03-25 22:53:58,166 - ThredUpScraper - WARNING - Saved Cloudflare debug artifacts to data\raw\scraped\debug_thredup
2026-03-25 22:53:58,443 - ThredUpScraper - INFO - Scraping page 2...
2026-03-25 22:53:58,481 - ThredUpScraper - INFO - GET https://www.thredup.com/search/dress?text=dress&page=2
2026-03-25 22:53:58,578 - ThredUpScraper - INFO - Status=403 Title=Just a moment...
2026-03-25 22:54:01,660 - ThredUpScraper - WARNING - Saved Cloudflare debug artifacts to data\raw\scraped\debug_thredup
2026-03-25 22:54:01,666 - ThredUpScraper - INFO - No more items found
2026-03-25 22:54:01,666 - ThredUpScraper - INFO - Completed scraping 0 items from ThredUp
2026-03-25 22:54:01,667 - DepopScraper


 Failed to scrape thredup: 'ThredUpScraper' object has no attribute '_browser_thread'

Starting scrape: DEPOP
Query: dress
Max items: 500
Rate limit: 2.0s between requests


2026-03-25 22:54:02,207 - DepopScraper - INFO - Playwright browser initialized with anti-bot detection avoidance
2026-03-25 22:54:02,430 - DepopScraper - WARNING - API scraping failed, falling back to web scraping
2026-03-25 22:54:04,284 - DepopScraper - INFO - Waiting for products to load (10 seconds)...
2026-03-25 22:54:16,328 - DepopScraper - INFO - Found 24 product links
2026-03-25 22:54:16,334 - DepopScraper - INFO - Scraping item: https://www.depop.com/products/brynbnoz-princess-polly-white-strapless-mini-2976/
2026-03-25 22:54:25,673 - DepopScraper - INFO - Scraping item: https://www.depop.com/products/chvoechea57-2x-dress-fits-like-large-e4e7/
2026-03-25 22:54:35,058 - DepopScraper - INFO - Scraping item: https://www.depop.com/products/blingraham813-cider-brown-multi-coloured-long-4b71/
2026-03-25 22:54:44,373 - DepopScraper - INFO - Scraping item: https://www.depop.com/products/cadencehmlt-altard-state-brown-floral-dress-d6db/
2026-03-25 22:54:53,785 - DepopScraper - INFO - Sc


 Failed to scrape depop: 'DepopScraper' object has no attribute '_browser_thread'

SCRAPING COMPLETE
Total items scraped: 0
  poshmark: 0 items
  thredup: 0 items
  depop: 0 items

Data saved to: data/raw/scraped
Starting scrape: POSHMARK
Query: pants
Max items: 500
Rate limit: 2.0s between requests


2026-03-25 22:58:01,270 - PoshmarkScraper - INFO - Playwright browser initialized
2026-03-25 22:58:05,751 - PoshmarkScraper - WARNING - Timeout waiting for selector: [class*="tile"], [class*="card"], [class*="item"]
2026-03-25 22:58:05,995 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Pants-69c20a96375218475123e19d
2026-03-25 22:58:11,474 - PoshmarkScraper - INFO - Successfully parsed item 69c20a96375218475123e19d: Pants
2026-03-25 22:58:11,475 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Pants-69c20a96375218475123e19d
2026-03-25 22:58:16,933 - PoshmarkScraper - INFO - Successfully parsed item 69c20a96375218475123e19d: Pants
2026-03-25 22:58:16,934 - PoshmarkScraper - INFO - Scraping item: https://poshmark.com/listing/Pants-69c20a96375218475123e19d
2026-03-25 22:58:22,428 - PoshmarkScraper - INFO - Successfully parsed item 69c20a96375218475123e19d: Pants
2026-03-25 22:58:22,430 - PoshmarkScraper - INFO - Scraping item: https://poshmark

## Clean And Organize

In [4]:
import clean_scraped_data
import organize_data

clean_scraped_data.main(['--input-dir', 'data/raw/scraped', '--output-dir', 'data/scraped', '--remove-no-images'])
organize_data.main(['--create-dirs'])

# this step takes up to 2 hours due to GBs being copied
organize_data.main(['--organize-deepfashion', '--copy'])
organize_data.main(['--verify'])
organize_data.main([ '--unify', '--format', 'both'])
organize_data.main([ '--summary', '--output', 'data/data_organization_report.json'])


Marketplace Data Cleaning & Normalization
Loading data from: data\raw\scraped
Loading depop...


  depop batches: 100%|██████████| 5/5 [00:00<00:00, 78.81it/s]


Loading poshmark...


  poshmark batches: 100%|██████████| 5/5 [00:00<00:00, 32.57it/s]


Loaded 4002 total items
Initial dataset: 4002 items
Normalizing platform-specific data to unified schema...


  Normalizing records: 100%|██████████| 4002/4002 [00:00<00:00, 15686.21it/s]
2026-03-26 00:05:47,414 - organize_data - INFO - Creating directory structure...
2026-03-26 00:05:47,415 - organize_data - INFO -  Created: data\deepfashion\original
2026-03-26 00:05:47,415 - organize_data - INFO -  Created: data\deepfashion\synthetic_degraded
2026-03-26 00:05:47,416 - organize_data - INFO -  Created: data\deepfashion\metadata
2026-03-26 00:05:47,416 - organize_data - INFO -  Created: data\scraped\poshmark
2026-03-26 00:05:47,417 - organize_data - INFO -  Created: data\scraped\thredup
2026-03-26 00:05:47,417 - organize_data - INFO -  Created: data\scraped\depop
2026-03-26 00:05:47,418 - organize_data - INFO -  Created: data\scraped\unified
2026-03-26 00:05:47,418 - organize_data - INFO -  Created: data\processed\train
2026-03-26 00:05:47,418 - organize_data - INFO -  Created: data\processed\val
2026-03-26 00:05:47,419 - organize_data - INFO -  Created: data\processed\test
2026-03-26 00:05:47,

Normalized 4002 records to unified schema
Removing duplicates...
Removed 2633 duplicates (65.8%)
Remaining items: 1369
Cleaning prices...
Removed 0 items with invalid prices
Price range: $3.00 - $950.00
Median price: $24.00
Cleaning text fields...
title: 1369/1369 (100.0%)
brand: 1257/1369 (91.8%)
category: 1368/1369 (99.9%)
condition: 1369/1369 (100.0%)
Standardizing conditions...
Standardized conditions:
Good: 905
New With Tags: 231
Like New: 198
New Without Tags: 32
Fair: 2
Fair Condition: 1
Validating image URLs...
Items with valid images: 1369/1369 (100.0%)
Filtering items without images...
Removed 0 items without images
Adding derived fields...
Saving cleaned unified data to: data\scraped
Saved unified CSV: data\scraped\unified_cleaned.csv
Saved full JSON: data\scraped\unified_cleaned.json
Saved depop: data\scraped\depop_cleaned.json (127 items)
Saved poshmark: data\scraped\poshmark_cleaned.json (1242 items)
Saved statistics: data\scraped\unified_stats.json
Unified Data Summary:


2026-03-26 00:05:47,667 - organize_data - INFO - Copied: data\raw\deepfashion\original\Anno -> data\deepfashion\Anno
2026-03-26 00:05:47,689 - organize_data - INFO - Copied: data\raw\deepfashion\original\Eval -> data\deepfashion\Eval
2026-03-26 00:35:55,436 - organize_data - INFO - Copied: data\raw\deepfashion\original\img -> data\deepfashion\img
2026-03-26 01:06:02,544 - organize_data - INFO - Copied: data\raw\deepfashion\original\img_highres -> data\deepfashion\img_highres
2026-03-26 01:06:02,560 - organize_data - INFO - Copied: data\raw\deepfashion\original\README.txt -> data\deepfashion\README.txt
2026-03-26 01:06:02,562 - organize_data - INFO - Found 3 source directories:
2026-03-26 01:06:02,562 - organize_data - INFO -   - kaggle_thushan
2026-03-26 01:06:02,563 - organize_data - INFO -   - kaggle_vishal
2026-03-26 01:06:02,563 - organize_data - INFO -   - original
2026-03-26 01:06:02,563 - organize_data - INFO - Copying kaggle_thushan... (this may take a while)
2026-03-26 01:55:4

## Prepare

In [ ]:
import generate_synthetic_data
import prepare_splits
import prepare_clothing_type_dataset
import prepare_condition_dataset
import prepare_multitask_dataset

#generate_synthetic_data.main(['--input-dir', SYNTHETIC_INPUT_DIR, '--output-dir', SYNTHETIC_OUTPUT_DIR, '--num-variations', str(SYNTHETIC_VARIATIONS), '--save-metadata', '--max-images', str(SYNTHETIC_MAX_IMAGES)])

# This steps creates separate Training, Validation and Testing sets. DeepFashion dataset already has its own separation. This split shuffles differently to mix in synthetic data.
# I've also discovered stratification as a method of fairer split regarding target category.
prepare_splits.main(['--all', '--clean', '--stratify', 'condition', '--report', 'data/processed/split_report.json'])
prepare_clothing_type_dataset.main(['--deepfashion-root', 'data/deepfashion', '--output-dir', CLOTHING_TYPE_DIR])
prepare_condition_dataset.main(['--deepfashion-dir', 'data/deepfashion/original', '--synthetic-dir', SYNTHETIC_OUTPUT_DIR, '--output-dir', CONDITION_DIR])
prepare_multitask_dataset.main(['--deepfashion2-csv', 'data/processed/deepfashion2_processed.csv', '--synthetic-root', SYNTHETIC_OUTPUT_DIR, '--output-dir', MULTITASK_DIR])


2026-03-26 02:27:29,148 - prepare_splits - INFO - Preparing directories for splits...
2026-03-26 02:27:29,148 - prepare_splits - WARNING - Removing existing directory: data\processed\train
2026-03-26 02:27:29,150 - prepare_splits - WARNING - Removing existing directory: data\processed\val
2026-03-26 02:27:29,151 - prepare_splits - WARNING - Removing existing directory: data\processed\test
2026-03-26 02:27:29,152 - prepare_splits - INFO - Directories prepared successfully!
2026-03-26 02:27:29,152 - prepare_splits - INFO - Splitting DeepFashion data...
2026-03-26 02:29:06,270 - prepare_splits - INFO - Found 899626 images to split
2026-03-26 02:29:06,271 - prepare_splits - INFO - Performing stratified split by: condition
2026-03-26 02:29:06,371 - prepare_splits - INFO - Found 1 groups for stratification
2026-03-26 02:29:07,247 - prepare_splits - INFO - Copying 629738 files to train set...


## Train Vision Models

In [ ]:
import train_clothing_type
import train_condition_assessment
import train_multitask_model

# first attempt to have individual models per task: cloth type
#train_clothing_type.main(['--data-dir', CLOTHING_TYPE_DIR, '--output-dir', CLOTHING_TYPE_MODEL_DIR, '--device', TRAIN_DEVICE])
# and condition
#train_condition_assessment.main(['--train-csv', f'{CONDITION_DIR}/train.csv', '--val-csv', f'{CONDITION_DIR}/val.csv', '--data-dir', str(PROJECT_ROOT), '--output-dir', CONDITION_MODEL_DIR])

history_path = r"models\condition_assessment\training_history.json"
with open(history_path, "r") as f:
    history = json.load(f)

df = pd.json_normalize(history)

plot_df = pd.concat(
    [
        df[["epoch", "train.loss", "train.mae", "train.rmse]]
        .rename(columns=lambda c: c.replace("train.", "") if c != "epoch" else c)
        .assign(split="Train"),
        df[["epoch", "val.loss", "val.mae", "val.rmse"]]
        .rename(columns=lambda c: c.replace("val.", "") if c != "epoch" else c)
        .assign(split="Validation"),
    ],
    ignore_index=True,
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
metrics = ["loss", "mae", "rmse"]
titles = ["Loss", "MAE", "RMSE"]

for ax, metric, title in zip(axes.flat, metrics, titles):
    sns.lineplot(
        data=plot_df,
        x="epoch",
        y=metric,
        hue="split",
        marker="o",
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)

plt.show()


# second attempt: train one vision model to categorize both cloth type and condition
#train_multitask_model.main(['--train-csv', f'{MULTITASK_DIR}/train.csv', '--val-csv', f'{MULTITASK_DIR}/val.csv', '--output-dir', MULTITASK_MODEL_DIR])

history_path = r"models\multitask\training_history.json"

with open(history_path, "r") as f:
    history = json.load(f)

df = pd.DataFrame({
    "epoch": range(1, len(history["train_loss"]) + 1),
    "train_loss": history["train_loss"],
    "val_loss": history["val_loss"],
    "train_accuracy": history["train_accuracy"],
    "val_accuracy": history["val_accuracy"],
    "train_mae": history["train_condition_mae"],
    "val_mae": history["val_condition_mae"],
})

plot_df = pd.concat(
    [
        df[["epoch", "train_loss", "train_accuracy", "train_mae"]]
        .rename(columns={
            "train_loss": "loss",
            "train_accuracy": "accuracy",
            "train_mae": "mae",
        })
        .assign(split="Train"),
        df[["epoch", "val_loss", "val_accuracy", "val_mae"]]
        .rename(columns={
            "val_loss": "loss",
            "val_accuracy": "accuracy",
            "val_mae": "mae",
        })
        .assign(split="Validation"),
    ],
    ignore_index=True,
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)
metrics = ["loss", "accuracy", "mae"]
titles = ["Loss", "Accuracy", "MAE"]

for ax, metric, title in zip(axes, metrics, titles):
    sns.lineplot(
        data=plot_df,
        x="epoch",
        y=metric,
        hue="split",
        marker="o",
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)

plt.show()

## Train Price Model

In [ ]:
from src.models.price import FeatureExtractor
from src.models.vision import MultiTaskClothingModel
from src.models.price import PriceRangeBinner


def get_config_value(config: dict, paths, default=None):
    for path in paths:
        value = config
        for key in path:
            if not isinstance(value, dict) or key not in value:
                break
            value = value[key]
        else:
            return value
    return default


# for PyTorch
class ClothingDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform, image_column: str = "image_path"):
        self.df = df
        self.transform = transform
        self.image_column = image_column

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = self.df.iloc[idx][self.image_column]
        image = Image.open(image_path).convert("RGB")
        return self.transform(image), idx


data_csv = Path(UNIFIED_DATA_CSV)
if not data_csv.exists():
    raise FileNotFoundError(data_csv)

vision_model_path = Path(VISION_MODEL_FOR_FEATURES)
if not vision_model_path.exists():
    raise FileNotFoundError(vision_model_path)

batch_size = 32
device = 'cuda'
output_dir = Path(FEATURES_DIR)

output_dir.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(data_csv)

required_cols = {"title", "description", "brand", "category", "condition", "price"}
missing_cols = sorted(required_cols - set(df.columns))
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# vision model
checkpoint = torch.load(vision_model_path, map_location=device)
config = checkpoint.get(
    "config",
    {
        "model": {
            "num_clothing_types": 20,
            "backbone": "efficientnet_b4",
            "condition_mode": "regression",
            "condition_scale": 10,
            "pretrained": False,
        }
    },
)
state_dict = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
classifier_weight = state_dict.get("type_classifier.4.weight")
inferred_num_classes = int(classifier_weight.shape[0]) if classifier_weight is not None and hasattr(classifier_weight, "shape") else None
vision_model = MultiTaskClothingModel(
    backbone_name=get_config_value(config, [("model", "backbone"), ("backbone",)], "efficientnet_b4"),
    num_clothing_types=int(
        get_config_value(
            config,
            [("model", "num_clothing_types"), ("num_clothing_types",), ("num_categories",)],
            inferred_num_classes or 20,
        )
    ),
    condition_scale=int(get_config_value(config, [("model", "condition_scale"), ("condition_scale",)], 10)),
    condition_mode=get_config_value(config, [("model", "condition_mode"), ("condition_mode",)], "regression"),
    pretrained=False,
    freeze_backbone=bool(get_config_value(config, [("model", "freeze_backbone"), ("freeze_backbone",)], False)),
)
vision_model.load_state_dict(state_dict)
vision_model.to(device)
vision_model.eval()

feature_extractor = FeatureExtractor(
    vision_model=vision_model,
    text_features=True,
    categorical_features=True,
    device=device,
)

vision_embeddings = None
if vision_model is not None:
    if "image_path" not in df.columns:
        raise ValueError("image_path column is required when vision embeddings are enabled")
    dataset = ClothingDataset(
        df,
        transform=transforms.Compose(
            [
                transforms.Resize((380, 380)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ]
        ),
    )
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=device == "cuda",
    )
    embeddings = []
    with torch.no_grad():
        for images, _ in tqdm(dataloader, desc="embeddings"):
            embeddings.append(feature_extractor.extract_vision_embeddings(images.to(device)))
    vision_embeddings = np.vstack(embeddings)
    np.save(output_dir / "vision_embeddings.npy", vision_embeddings)

data_dict = {
    "title": df["title"].fillna("").astype(str).tolist(),
    "description": df["description"].fillna("").astype(str).tolist(),
    "brand": df["brand"].fillna("unknown").astype(str).tolist(),
    "category": df["category"].fillna("unknown").astype(str).tolist(),
    "condition": df["condition"].fillna("good").astype(str).tolist(),
    "condition_score": df.get("condition_score", pd.Series([5.0] * len(df))).fillna(5.0).values,
}
features, feature_names = feature_extractor.extract_all_features(data_dict, images=None, fit=True)

np.save(output_dir / "features.npy", features)
with open(output_dir / "feature_names.json", "w") as f:
    json.dump(feature_names, f, indent=2)
feature_extractor.save(output_dir / "feature_extractor.pkl")

metadata = {
    "n_samples": int(features.shape[0]),
    "n_features": int(features.shape[1]),
    "has_vision_embeddings": vision_embeddings is not None,
    "vision_embedding_dim": int(vision_embeddings.shape[1]) if vision_embeddings is not None else 0,
    "feature_names": feature_names,
    "data_csv": str(data_csv),
    "vision_model": str(vision_model_path) if vision_model_path else None,
}
with open(output_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

DEFAULT_BINNING_STRATEGY = 'quantile'
DEFAULT_BIN_COUNT = 5
DEFAULT_TRAIN_RATIO = 0.7
DEFAULT_VAL_RATIO = 0.15
DEFAULT_TEST_RATIO = 0.15
DEFAULT_RANDOM_SEED = 42

def make_json_serializable(value):
    if isinstance(value, dict):
        return {str(k): make_json_serializable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_serializable(v) for v in value]
    if isinstance(value, np.ndarray):
        return [make_json_serializable(v) for v in value.tolist()]
    if isinstance(value, np.generic):
        return value.item()
    return value

data_csv = Path(UNIFIED_DATA_CSV)
output_dir = Path(PRICE_DATASET_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
    
if not data_csv.exists():
    raise FileNotFoundError(f"Data CSV not found: {data_csv}")
    
df = pd.read_csv(data_csv)
if 'price' not in df.columns:
    raise ValueError("'price' column not found in data CSV")
    
prices = df['price'].values

stats = {
    'count': int(len(prices)),
    'mean': float(np.mean(prices)),
    'median': float(np.median(prices)),
    'std': float(np.std(prices)),
    'min': float(np.min(prices)),
    'max': float(np.max(prices)),
    'q25': float(np.percentile(prices, 25)),
    'q75': float(np.percentile(prices, 75))
}

print(f"Price Statistics:")
print(f"Count: ${stats['count']}")
print(f"Mean: ${stats['mean']:.2f}")
print(f"Median: ${stats['median']:.2f}")
print(f"Std Dev: ${stats['std']:.2f}")
print(f"Min: ${stats['min']:.2f}")
print(f"Max: ${stats['max']:.2f}")
print(f"Q25: ${stats['q25']:.2f}")
print(f"Q75: ${stats['q75']:.2f}")

if category_col in df.columns:
    category_stats = df.groupby(category_col)['price'].agg([
        'count', 'mean', 'median', 'std', 'min', 'max'
    ]).round(2)
    print(f"Price by Category:")
    print(category_stats)
    
    category_stats.to_csv(output_dir / 'price_by_category.csv')

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

axes[0, 0].hist(prices, bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Price Distribution')
axes[0, 0].axvline(stats['mean'], color='red', linestyle='--', label=f"Mean: ${stats['mean']:.2f}")
axes[0, 0].axvline(stats['median'], color='green', linestyle='--', label=f"Median: ${stats['median']:.2f}")
axes[0, 0].legend()

axes[0, 1].hist(prices, bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Price ($)')
axes[0, 1].set_ylabel('Frequency (log scale)')
axes[0, 1].set_title('Price Distribution (Log Scale)')
axes[0, 1].set_yscale('log')

axes[1, 0].boxplot(prices, vert=True)
axes[1, 0].set_ylabel('Price ($)')
axes[1, 0].set_title('Price Box Plot')
axes[1, 0].grid(axis='y', alpha=0.3)

if category_col in df.columns:
    categories = df[category_col].unique()[:10]  # Limit to 10 categories for readability
    category_prices = [df[df[category_col] == cat]['price'].values for cat in categories]
    axes[1, 1].boxplot(category_prices, labels=categories, vert=True)
    axes[1, 1].set_xlabel('Category')
    axes[1, 1].set_ylabel('Price ($)')
    axes[1, 1].set_title('Price by Category (Top 10)')
    axes[1, 1].tick_params(axis='x', rotation=45)
else:
    axes[1, 1].text(0.5, 0.5, 'Category data not available', 
                   ha='center', va='center', transform=axes[1, 1].transAxes)

plt.tight_layout()
plt.savefig(output_dir / 'price_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

print(f"Saved price distribution plot to {output_dir / 'price_distribution.png'}")

# Save statistics
with open(output_dir / 'price_statistics.json', 'w') as f:
    json.dump(stats, f, indent=2)

prices = df['price'].values
categories = None

binner = PriceRangeBinner(
    strategy=DEFAULT_BINNING_STRATEGY,
    n_bins=DEFAULT_BIN_COUNT,
    custom_bins=None,
    category_specific=False
)

bin_indices, bin_labels = binner.fit_transform(prices, categories)

df['price_bin'] = bin_indices
df['price_bin_label'] = bin_labels

print(f"Price Bins:")
bin_info = binner.get_bin_info()
for cat, bins in bin_info['bins'].items():
    labels = bin_info['labels'][cat]
    print(f"Category: {cat}")
    for i, (label, bin_start, bin_end) in enumerate(zip(labels, bins[:-1], bins[1:])):
        count = np.sum((bin_indices == i) & ((categories == cat) if categories is not None else True))
        print(f"Bin {i}: {label} (n={count})")

fig, ax = plt.subplots(figsize=(12, 6))

bin_counts = pd.Series(bin_labels).value_counts().sort_index()
ax.bar(range(len(bin_counts)), bin_counts.values, edgecolor='black', alpha=0.7)
ax.set_xlabel('Price Bin')
ax.set_ylabel('Number of Samples')
ax.set_title('Distribution of Samples Across Price Bins')
ax.set_xticks(range(len(bin_counts)))
ax.set_xticklabels(bin_counts.index, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

for i, v in enumerate(bin_counts.values):
    ax.text(i, v + max(bin_counts.values) * 0.01, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.savefig(output_dir / 'bin_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

serializable_bin_info = make_json_serializable(bin_info)
with open(output_dir / 'price_binner.json', 'w') as f:
    json.dump(serializable_bin_info, f, indent=2)

train_ratio: float = DEFAULT_TRAIN_RATIO,
val_ratio: float = DEFAULT_VAL_RATIO,
test_ratio: float = DEFAULT_TEST_RATIO,
random_seed: int = DEFAULT_RANDOM_SEED
if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
    raise ValueError("Split ratios must sum to 1.0")

X_trainval, X_test, y_trainval, y_test, idx_trainval, idx_test = train_test_split(
    features, labels, np.arange(len(labels)),
    test_size=test_ratio,
    stratify=labels,
    random_state=random_seed
)

val_size = val_ratio / (train_ratio + val_ratio)
X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_trainval, y_trainval, idx_trainval,
    test_size=val_size,
    stratify=y_trainval,
    random_state=random_seed
)

splits = {
    'train': (X_train, y_train, idx_train),
    'val': (X_val, y_val, idx_val),
    'test': (X_test, y_test, idx_test)
}

for split_name, (X, y, idx) in splits.items():
    np.save(output_dir / f'{split_name}_features.npy', X)
    np.save(output_dir / f'{split_name}_labels.npy', y)
    np.save(output_dir / f'{split_name}_indices.npy', idx)
    
    split_df = df.iloc[idx].copy()
    split_df.to_csv(output_dir / f'{split_name}_data.csv', index=False)
    
    print(f"{split_name.capitalize()} split: {len(X)} samples")
    
    unique, counts = np.unique(y, return_counts=True)
    print(f"Label distribution: {dict(zip(unique, counts))}")

metadata = {
    'train_ratio': train_ratio,
    'val_ratio': val_ratio,
    'test_ratio': test_ratio,
    'random_seed': random_seed,
    'n_train': int(len(X_train)),
    'n_val': int(len(X_val)),
    'n_test': int(len(X_test)),
    'n_features': int(features.shape[1]),
    'n_classes': int(len(np.unique(labels)))
}

with open(output_dir / 'split_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

X_train = np.load(PRICE_DATASET_DIR / 'train_features.npy')
y_train = np.load(PRICE_DATASET_DIR / 'train_labels.npy')
X_val = np.load(PRICE_DATASET_DIR / 'val_features.npy')
y_val = np.load(PRICE_DATASET_DIR / 'val_labels.npy')
X_test = np.load(PRICE_DATASET_DIR / 'test_features.npy')
y_test = np.load(PRICE_DATASET_DIR / 'test_labels.npy')

with open(PRICE_DATASET_DIR / 'price_binner.json', 'r') as f:
    binner_info = json.load(f)

with open(PRICE_DATASET_DIR / 'split_metadata.json', 'r') as f:
    split_info = json.load(f)

feature_names = None
feature_names_path = PRICE_DATASET_DIR.parent / 'features' / 'feature_names.json'
if feature_names_path.exists():
    with open(feature_names_path, 'r') as f:
        feature_names = json.load(f)

bins_dict = binner_info.get('bins', {})
labels_dict = binner_info.get('labels', {})

if 'global' in labels_dict:
    class_names = labels_dict['global']
else:
    first_cat = list(labels_dict.keys())[0]
    class_names = labels_dict[first_cat]

print(f"Train: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Val: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
print(f"Classes: {len(class_names)} price ranges")
print(f"Class names: {class_names}")

data = {
    'X_train': X_train,
    'y_train': y_train,
    'X_val': X_val,
    'y_val': y_val,
    'X_test': X_test,
    'y_test': y_test,
    'class_names': class_names,
    'feature_names': feature_names,
    'n_classes': len(class_names),
    'binner_info': binner_info,
    'split_info': split_info
}
    
output_dir = Path(PRICE_MODEL_DIR)

models_config = {
    'xgboost': {
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.1,
        'random_state': 42
    },
    'lightgbm': {
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.1,
        'random_state': 42,
        'verbose': -1
    },
    'random_forest': {
        'n_estimators': 100,
        'max_depth': 10,
        'random_state': 42,
        'n_jobs': -1
    }
}

results = []

for model_type, params in models_config.items():
    try:
        result = train_model(model_type, data, params, verbose=False)
        results.append(result)
    except Exception as e:
        print(f"Error training {model_type}: {e}")
        continue

print("MODEL COMPARISON RESULTS")

comparison_data = []
for result in results:
    comparison_data.append({
        'Model': result['model_type'].upper(),
        'Train Time (s)': f"{result['train_time']:.2f}",
        'Val Accuracy': f"{result['val_metrics']['accuracy']:.4f}",
        'Val F1': f"{result['val_metrics']['f1_weighted']:.4f}",
        'Val +/-1': f"{result['val_metrics']['within_1_accuracy']:.4f}",
        'Test Accuracy': f"{result['test_metrics']['accuracy']:.4f}",
        'Test F1': f"{result['test_metrics']['f1_weighted']:.4f}",
        'Test +/-1': f"{result['test_metrics']['within_1_accuracy']:.4f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

comparison_df.to_csv(output_dir / 'model_comparison.csv', index=False)
print(f"Saved comparison to {output_dir / 'model_comparison.csv'}")

best_result = max(results, key=lambda x: x['test_metrics']['accuracy'])
print(f"Best Model: {best_result['model_type'].upper()}")
print(f"Test Accuracy: {best_result['test_metrics']['accuracy']:.4f}")
print(f"Test F1: {best_result['test_metrics']['f1_weighted']:.4f}")
print(f"Within +/-1 bracket: {best_result['test_metrics']['within_1_accuracy']:.4f}")


## Generate Embeddings

In [ ]:
from src.models.embeddings import (
    VisionEmbeddingExtractor,
    TextEmbeddingExtractor,
    MultiModalEmbedding,
    EmbeddingPipeline,

df = pd.read_csv(UNIFIED_DATA_CSV)
images, texts = prepare_data(df, args)

print("INITIALIZING EMBEDDING PIPELINE")
pipeline = EmbeddingPipeline(
    vision_model_path=VISION_MODEL_FOR_FEATURES,
    text_model_name=None,
    fusion_method='concat',
    reduce_dim=True,
    reduction_method='pca',
    target_dim=128,
    device=TRAIN_DEVICE,
)

print(f"Embedding dimension: {pipeline.multi_modal.embedding_dim}")
print(f"Will be reduced to: {args.target_dim}")

print("GENERATING EMBEDDINGS")

embeddings = pipeline.generate(
    images=images,
    texts=texts,
    batch_size=32,
    normalize=True,
)

print(f"Generated embeddings shape: {embeddings.shape}")

metadata = {
    'mode': 'multimodal',
    'num_samples': len(embeddings),
    'embedding_dim': embeddings.shape[1],
    'vision_model': VISION_MODEL_FOR_FEATURES,
    'text_model': None,
    'fusion_method': 'concat',
    'reduced': True,
    'reduction_method': 'pca',
    'target_dim': 128,
    'normalized': True,
    'generated_at': datetime.now().isoformat(),
    'data_source': UNIFIED_DATA_CSV,
    'item_ids': df.get('item_id', df.index).tolist(),
}

print("SAVING EMBEDDINGS")

pipeline.save(EMBEDDINGS_DIR, embeddings, metadata)

df_path = Path(EMBEDDINGS_DIR) / 'items_data.csv'
df.to_csv(df_path, index=False)


## Build Vector Index

In [ ]:
from src.vector_search import FAISSIndex, SimilaritySearch

embeddings_dir = Path(EMBEDDINGS_DIR)
embeddings_path = embeddings_dir / 'embeddings.npy'
if not embeddings_path.exists():
    raise FileNotFoundError(f"Embeddings not found at {embeddings_path}")

embeddings = np.load(str(embeddings_path))
print(f"Loaded embeddings: shape={embeddings.shape}")

metadata_path = embeddings_dir / 'metadata.json'
if metadata_path.exists():
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    item_ids = metadata.get('item_ids', [])
else:
    item_ids = list(range(len(embeddings)))

items_csv = embeddings_dir / 'items_data.csv'
if items_csv.exists():
    items_data = pd.read_csv(items_csv)
    print(f"Loaded items data: {len(items_data)} items")
else:
    items_data = pd.DataFrame({'item_id': item_ids})
    print("Warning: items_data.csv not found, using minimal item IDs")

index_metadata = [{'item_id': item_id} for item_id in item_ids]

print(f"Building {index_type.upper()} index...")
print(f"Metric: {metric}")
print(f"Embeddings: {embeddings.shape}")

start_time = time.time()
index_type = 'flat'
metric = 'cosine'
nlist = 100
nprobe = 10
hnsw_m = 32
use_gpu = True

index = FAISSIndex(
    index_type=index_type,
    dimension=embeddings.shape[1],
    metric=metric,
    nlist=nlist,
    nprobe=nprobe,
    hnsw_m=hnsw_m,
    use_gpu=use_gpu,
)
index.build(embeddings, metadata)

build_time = time.time() - start_time
print(f"Index built in {build_time:.2f} seconds")

stats = index.get_stats()
print("Index Statistics:")
for key, value in stats.items():
    print(f"{key}: {value}")

output_dir = Path(VECTOR_INDEX_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'clothing_index'
index.save(str(output_path))
items_csv_path = output_dir / 'items_data.csv'
items_data.to_csv(items_csv_path, index=False)
print(f"Saved items data to {items_csv_path}")
print(f"Index saved to {output_path}")
